# Die Kaggle-Grandmaster Strategie: Ensembling (LightGBM + CatBoost)

Wir sind bei **0.7828** – es fehlt nur noch ein winziger Schritt bis zur 0.80!
Wenn ein einzelnes Modell (LightGBM) nicht reicht, nutzen Profis bei Kaggle **Ensembles**. 

Wir trainieren jetzt **gleichzeitig zwei der besten Algorithmen der Welt**:
1. **LightGBM** (das kennst du schon)
2. **CatBoost** (ein extrem mächtiger Algorithmus von Yandex, der oft noch besser abschneidet)

Am Ende lassen wir beide Modelle abstimmen (wir mitteln ihre Wahrscheinlichkeiten). Da CatBoost andere Fehler macht als LightGBM, gleicht sich das perfekt aus und gibt oft einen **kostenlosen Boost von 1-2 %**!

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from scipy.stats import skew, kurtosis
import warnings
warnings.filterwarnings('ignore')

## 1. Daten & Feature Extraction (Unsere bewährten Ultimate-Features)

In [ ]:
KAGGLE_PATH = '/kaggle/input/datasets/axxtur/nycu-data-mining-assignment-3'
if not os.path.exists(KAGGLE_PATH):
    KAGGLE_PATH = '/kaggle/input/nycu-data-mining-assignment-3'
    if not os.path.exists(KAGGLE_PATH):
        KAGGLE_PATH = 'nycu-data-mining-assignment-3'

train_data = np.load(os.path.join(KAGGLE_PATH, 'train_data.npz'), allow_pickle=True)
X_train_raw = train_data['X']
y_train = train_data['y']
file_ids_train = train_data['file_ids']
user_ids_train = train_data['user_ids']

test_data = np.load(os.path.join(KAGGLE_PATH, 'test_data.npz'), allow_pickle=True)
X_test_raw = test_data['X']
file_ids_test = test_data['file_ids']

def extract_features_np(X):
    def calc_stats(arr, axis):
        f = []
        f.append(np.mean(arr, axis=axis))
        f.append(np.std(arr, axis=axis))
        f.append(np.min(arr, axis=axis))
        f.append(np.max(arr, axis=axis))
        f.append(np.median(arr, axis=axis))
        f.append(np.percentile(arr, 25, axis=axis))
        f.append(np.percentile(arr, 75, axis=axis))
        f.append(skew(arr, axis=axis))
        f.append(kurtosis(arr, axis=axis))
        f.append(np.max(arr, axis=axis) - np.min(arr, axis=axis))
        f.append(np.sum(arr**2, axis=axis))
        diffs = np.diff(arr, axis=axis)
        f.append(np.mean(np.abs(diffs), axis=axis))
        f.append(np.std(diffs, axis=axis))
        fft_vals = np.abs(np.fft.rfft(arr, axis=axis))
        f.append(np.mean(fft_vals, axis=axis))
        f.append(np.std(fft_vals, axis=axis))
        f.append(np.max(fft_vals, axis=axis))
        def calc_corr(a, b, ax):
            a_mean = np.mean(a, axis=ax, keepdims=True)
            b_mean = np.mean(b, axis=ax, keepdims=True)
            a_std = np.std(a, axis=ax, keepdims=True)
            b_std = np.std(b, axis=ax, keepdims=True)
            cov = np.mean((a - a_mean) * (b - b_mean), axis=ax, keepdims=True)
            return np.squeeze(cov / (a_std * b_std + 1e-8), axis=ax)
        f.append(calc_corr(arr[..., 0], arr[..., 1], axis))
        f.append(calc_corr(arr[..., 0], arr[..., 2], axis))
        f.append(calc_corr(arr[..., 1], arr[..., 2], axis))
        mag = np.sqrt(arr[..., 0]**2 + arr[..., 1]**2 + arr[..., 2]**2)
        f.append(np.mean(mag, axis=axis, keepdims=True))
        f.append(np.std(mag, axis=axis, keepdims=True))
        f.append(np.max(mag, axis=axis, keepdims=True))
        return f
    global_feats = [f.reshape(X.shape[0], -1) for f in calc_stats(X, axis=1)]
    sub_feats = [f.reshape(X.shape[0], -1) for f in calc_stats(X.reshape(X.shape[0], 5, 60, 6), axis=2)]
    return np.concatenate(global_feats + sub_feats, axis=1)

print("Extrahiere Features...")
X_train_feat = extract_features_np(X_train_raw)
X_test_feat = extract_features_np(X_test_raw)


## 2. Das Ensemble Training (LGBM + CatBoost)

In [ ]:
gkf = GroupKFold(n_splits=5)
models_lgb = []
models_cb = []
scores_ensemble = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_feat, y_train, groups=user_ids_train)):
    X_tr, X_va = X_train_feat[train_idx], X_train_feat[val_idx]
    y_tr, y_va = y_train[train_idx], y_train[val_idx]
    
    print(f"\n{'='*10} FOLD {fold+1} {'='*10}")
    
    # 1. Trainiere LightGBM
    print("Trainiere LightGBM...")
    clf_lgb = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        max_depth=7,
        num_leaves=64,
        verbose=-1
    )
    clf_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)])
    
    # 2. Trainiere CatBoost
    print("Trainiere CatBoost...")
    clf_cb = cb.CatBoostClassifier(
        iterations=1000,
        learning_rate=0.03,
        depth=6,
        auto_class_weights='Balanced',
        random_state=42,
        verbose=False,
        early_stopping_rounds=50
    )
    clf_cb.fit(X_tr, y_tr, eval_set=(X_va, y_va))
    
    # Wahrscheinlichkeiten beider Modelle vorhersagen
    preds_lgb = clf_lgb.predict_proba(X_va)
    preds_cb = clf_cb.predict_proba(X_va)
    
    # ENSEMBLE: Einfacher Durchschnitt der Wahrscheinlichkeiten
    preds_ensemble = (preds_lgb + preds_cb) / 2.0
    
    # Welches Label hat die höchste gemittelte Wahrscheinlichkeit?
    final_labels = np.argmax(preds_ensemble, axis=1)
    
    fold_f1 = f1_score(y_va, final_labels, average='macro')
    print(f"-> Fold {fold+1} Ensemble F1-Macro: {fold_f1:.4f}")
    
    scores_ensemble.append(fold_f1)
    models_lgb.append(clf_lgb)
    models_cb.append(clf_cb)

print(f"\nOverall Cross-Validation ENSEMBLE F1-Macro: {np.mean(scores_ensemble):.4f}")

## 3. Test Predictions und Kaggle Submission

In [ ]:
test_preds_proba = np.zeros((len(X_test_feat), 6))

for clf_lgb, clf_cb in zip(models_lgb, models_cb):
    # Wir summieren die Wahrscheinlichkeiten beider Modelle über alle 5 Folds auf
    lgb_prob = clf_lgb.predict_proba(X_test_feat)
    cb_prob = clf_cb.predict_proba(X_test_feat)
    
    fold_ensemble_prob = (lgb_prob + cb_prob) / 2.0
    test_preds_proba += fold_ensemble_prob / 5.0 # Durchschnitt über die 5 Folds

final_preds = np.argmax(test_preds_proba, axis=1)

submission = pd.DataFrame({
    'Id': file_ids_test,
    'Label': final_preds
})

submission.to_csv('submission_ensemble.csv', index=False)
print("Saved submission_ensemble.csv! Ready for Kaggle upload.")
submission.head()